# HW7 — Full Python Solution

*Fill in `PARENT_DIR` and `MAT_FILE` at the bottom, then run all.*

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

In [2]:

"""
HW7 — Full Python Solution
Author: <Your Name>
Notes:
- Edit PARENT_DIR and MAT_FILE to point to your dataset.
- Assumes the .mat contains keys 'Signal' (trials × time × channels) and 'Type' (length trials),
  where Type==1 marks Target, Type==0 marks Non-target. If different, adjust EXTRACT_KEYS.
"""

from __future__ import annotations
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Try both SciPy and h5py loader to support MATLAB v7.3 files
def load_mat_generic(mat_path: Path, keys=('Signal','Type')) -> dict:
    """
    Load MATLAB .mat (v7.3 or earlier) and return a dictionary containing at least the requested keys.
    - If v7.3 (HDF5), we use h5py to read datasets.
    - Else we use scipy.io.loadmat.
    """
    mat_path = Path(mat_path)
    assert mat_path.exists(), f"MAT file not found: {mat_path}"
    data = {}
    try:
        # First try scipy.io for non-HDF5
        import scipy.io as sio
        raw = sio.loadmat(mat_path.as_posix(), squeeze_me=True, struct_as_record=False)
        # Flatten MATLAB structs/dtypes into Pythonic arrays where possible
        for k in keys:
            if k in raw:
                data[k] = np.array(raw[k])
    except Exception:
        # Fall back to h5py for v7.3
        import h5py
        with h5py.File(mat_path.as_posix(), "r") as f:
            for k in keys:
                if k in f:
                    arr = np.array(f[k])  # MATLAB HDF5 is column-major
                    data[k] = np.array(arr).squeeze()
    # Sanity check
    missing = [k for k in keys if k not in data]
    if missing:
        raise KeyError(f"Could not find keys {missing} in {mat_path.name}. Available keys may differ. "
                       f"Open the file and inspect keys to adjust 'keys='.")
    return data

# ---------------------------
# 0) Global constants
# ---------------------------
# Band-pass (not used in plotting directly but kept per instructions)
bp_low = 0.5
bp_upp = 6
electrode_num = 16

# ---------------------------
# 1) Directories
# ---------------------------
def ensure_dirs(parent: Path) -> dict[str, Path]:
    """
    Create 'exports' and 'figures' under parent if missing.
    Returns dict of created paths.
    """
    parent = Path(parent)
    parent.mkdir(parents=True, exist_ok=True)
    paths = {
        "parent": parent,
        "exports": parent / "exports",
        "figures": parent / "figures",
    }
    for p in paths.values():
        p.mkdir(exist_ok=True, parents=True)
    return paths

# ---------------------------
# 2) Helpers: indexing & labels
# ---------------------------
def default_channel_names(n_channels: int) -> list[str]:
    # Generic 16-channel example list; replace if you have your own channel map.
    fallback = ["F3","F4","C3","C4","CP3","CP4","P3","P4","Fp1","Fp2","F7","F8","T3","T4","O1","O2"]
    if n_channels <= len(fallback):
        return fallback[:n_channels]
    return [f"Ch{c+1}" for c in range(n_channels)]

def time_axis(n_time: int, Fs: float | None = None) -> np.ndarray:
    """
    Build a time axis in ms. If Fs (Hz) is provided, use it; else just index in ms.
    """
    if Fs and Fs > 0:
        t = np.arange(n_time) / Fs * 1000.0  # ms
    else:
        t = np.arange(n_time)  # treat as ms index
    return t

# ---------------------------
# 3) Compute type-specific means & covariance
# ---------------------------
def compute_stats_per_channel(signal: np.ndarray, types: np.ndarray) -> dict:
    """
    Inputs:
      signal: array (n_trials, n_time, n_channels)
      types:  array (n_trials,) with 1=Target, 0=Non-target (or boolean)
    Returns:
      dict with 'means' and 'covs', each a dict with keys 'all','target','non_target'.
      For 'means': array shape (n_time, n_channels).
      For 'covs' : list length n_channels of (n_time × n_time) covariance matrices.
    """
    assert signal.ndim == 3, "Expected signal with shape (trials, time, channels)."
    n_trials, n_time, n_channels = signal.shape

    # Masks
    tmask = (types == 1) | (types == True)
    nmask = (types == 0) | (types == False)

    def _mean_over_trials(X):  # (trials, time, channels) -> (time, channels)
        return X.mean(axis=0)

    def _cov_time_by_time(X):  # (trials, time) -> (time, time)
        # Each row is a trial; set rowvar=False to treat columns (time) as variables
        return np.cov(X, rowvar=False, bias=False)

    # All
    mean_all = _mean_over_trials(signal)
    cov_all = [ _cov_time_by_time(signal[:,:,ch]) for ch in range(n_channels) ]

    # Target
    sig_t = signal[tmask]
    mean_t = _mean_over_trials(sig_t) if sig_t.size else np.full((n_time, n_channels), np.nan)
    cov_t  = [ _cov_time_by_time(sig_t[:,:,ch]) if sig_t.size else np.full((n_time,n_time), np.nan)
               for ch in range(n_channels) ]

    # Non-target
    sig_n = signal[nmask]
    mean_n = _mean_over_trials(sig_n) if sig_n.size else np.full((n_time, n_channels), np.nan)
    cov_n  = [ _cov_time_by_time(sig_n[:,:,ch]) if sig_n.size else np.full((n_time,n_time), np.nan)
               for ch in range(n_channels) ]

    return {
        "means": {"all": mean_all, "target": mean_t, "non_target": mean_n},
        "covs":  {"all": cov_all,  "target": cov_t,  "non_target": cov_n},
    }

# ---------------------------
# 4) Plotters
# ---------------------------
def plot_means(means: np.ndarray, title_prefix: str, channels: list[str], outpath: Path | None = None):
    """
    Plot mean (time series) per channel in a 4×4 grid (assuming 16 channels).
    means: (n_time, n_channels)
    """
    n_time, n_channels = means.shape
    assert n_channels == len(channels), "channels length mismatch"
    t = time_axis(n_time)
    rows = int(np.ceil(n_channels/4))
    cols = 4
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3.2*rows), squeeze=False)
    for ch in range(n_channels):
        r, c = divmod(ch, cols)
        ax = axes[r][c]
        ax.plot(t, means[:,ch])
        ax.set_title(channels[ch])
        ax.set_xlabel("Time (ms)")
        ax.set_ylabel("Amplitude (µV)")
    # Hide any empty subplots
    for ch in range(n_channels, rows*cols):
        r, c = divmod(ch, cols)
        axes[r][c].axis('off')
    fig.suptitle(f"{title_prefix} — Sample Mean by Channel", y=0.995, fontsize=14)
    fig.tight_layout()
    if outpath:
        fig.savefig(outpath, dpi=200, bbox_inches="tight")
    plt.show()

def plot_covariances(cov_list: list[np.ndarray], title_prefix: str, channels: list[str], outpath: Path | None = None):
    """
    Plot time×time covariance per channel using contourf in a 4×4 grid.
    cov_list: list length n_channels with (n_time × n_time) matrices.
    """
    n_channels = len(cov_list)
    n_time = cov_list[0].shape[0]
    t = time_axis(n_time)
    T1, T2 = np.meshgrid(t, t)  # for contourf
    rows = int(np.ceil(n_channels/4))
    cols = 4
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3.2*rows), squeeze=False)
    # find a common vmin/vmax for better comparability
    all_vals = np.concatenate([cov.ravel() for cov in cov_list if cov is not None])
    vmin, vmax = np.nanpercentile(all_vals, [2, 98])
    for ch in range(n_channels):
        r, c = divmod(ch, cols)
        ax = axes[r][c]
        cov = cov_list[ch]
        im = ax.contourf(T1, T2, cov, levels=20, vmin=vmin, vmax=vmax)
        ax.set_title(channels[ch])
        ax.set_xlabel("Time (ms)")
        ax.set_ylabel("Time (ms)")
    # Hide any empty subplots
    for ch in range(n_channels, rows*cols):
        r, c = divmod(ch, cols)
        axes[r][c].axis('off')
    # One colorbar for the whole figure
    cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85)
    cbar.set_label("Covariance (µV²)")
    fig.suptitle(f"{title_prefix} — Sample Covariance by Channel", y=0.995, fontsize=14)
    fig.tight_layout()
    if outpath:
        fig.savefig(outpath, dpi=200, bbox_inches="tight")
    plt.show()

# ---------------------------
# 5) Orchestrator for a participant
# ---------------------------
def analyze_participant(mat_file: str | Path,
                        parent_dir: str | Path,
                        participant_label: str = "",
                        channel_names: list[str] | None = None):
    """
    Full pipeline for one participant .mat file:
     - create dirs
     - load Signal & Type
     - compute stats
     - save 3 mean plots and 3 covariance plots (Target, Non-target, All)
    """
    paths = ensure_dirs(parent_dir)
    mat_file = Path(mat_file)
    # Load (assuming 'Signal' and 'Type')
    data = load_mat_generic(mat_file, keys=('Signal','Type'))

    # Coerce shapes: expect (trials, time, channels)
    signal = np.array(data['Signal'])
    types  = np.array(data['Type']).astype(int).ravel()
    if signal.ndim == 2:
        # If the .mat is (time, channels) but multiple trials are stacked elsewhere, user must adapt.
        raise ValueError("Expected signal to be 3D (trials, time, channels). Got 2D.")
    n_trials, n_time, n_channels = signal.shape

    # Channels
    channels = channel_names if channel_names is not None else default_channel_names(n_channels)

    # Compute stats
    stats = compute_stats_per_channel(signal, types)

    # Plot MEANS
    plot_means(stats["means"]["target"],     f"{participant_label} — Target",     channels,
               outpath=paths["figures"] / f"{participant_label}_means_target.png")
    plot_means(stats["means"]["non_target"], f"{participant_label} — Non-target", channels,
               outpath=paths["figures"] / f"{participant_label}_means_nontarget.png")
    plot_means(stats["means"]["all"],        f"{participant_label} — All",        channels,
               outpath=paths["figures"] / f"{participant_label}_means_all.png")

    # Plot COVARIANCES
    plot_covariances(stats["covs"]["target"],     f"{participant_label} — Target",     channels,
                     outpath=paths["figures"] / f"{participant_label}_cov_target.png")
    plot_covariances(stats["covs"]["non_target"], f"{participant_label} — Non-target", channels,
                     outpath=paths["figures"] / f"{participant_label}_cov_nontarget.png")
    plot_covariances(stats["covs"]["all"],        f"{participant_label} — All",        channels,
                     outpath=paths["figures"] / f"{participant_label}_cov_all.png")

    # Optionally, export some arrays for grading
    np.savez(paths["exports"] / f"{participant_label}_stats_arrays.npz",
             mean_target=stats["means"]["target"],
             mean_nontarget=stats["means"]["non_target"],
             mean_all=stats["means"]["all"],
             # Covariances are lists of matrices; save as a single 3D array per type if consistent
             cov_target=np.stack(stats["covs"]["target"], axis=-1),
             cov_nontarget=np.stack(stats["covs"]["non_target"], axis=-1),
             cov_all=np.stack(stats["covs"]["all"], axis=-1))

if __name__ == "__main__":
    # --------- EDIT THESE TWO LINES ---------
    PARENT_DIR = Path.home() / "GitHub" / "BIOS-584" / "HW7"     # change to your parent folder
    MAT_FILE   = PARENT_DIR / "data" / "K114.mat"                 # change to your actual .mat path
    # ---------------------------------------

    # Run analysis for participant K114
    try:
        analyze_participant(MAT_FILE, PARENT_DIR, participant_label="K114")
    except Exception as e:
        print(f"[Note] Run-time error (likely due to missing file or different key names): {e}")
        print("The code is complete; please set PARENT_DIR and MAT_FILE and re-run locally.")


[Note] Run-time error (likely due to missing file or different key names): MAT file not found: C:\Users\josep\GitHub\BIOS-584\HW7\data\K114.mat
The code is complete; please set PARENT_DIR and MAT_FILE and re-run locally.
